In [ ]:

# ── Town-Center Additional Jobs ──────────────────────────────────────────────
# 1. Convert TAZ and town-center polygons to in-memory feature classes
# 2. Spatial join → TAZs that intersect a town center
# 3. Load Alt-1 and Alt-2-reduced SocioEcon files (2050 only)
# 4. Compare residential units (Alt2 − Alt1) for town-center TAZs only
# 5. Distribute 176 additional jobs proportionally (integer), randomly
#    assign remainders until exactly 176 is reached; write CSV

import pathlib
import numpy as np
import pandas as pd
import arcpy
from utils import get_fs_data, get_fs_data_spatial
taz_url = r'https://maps.trpa.org/server/rest/services/Transportation_Planning/MapServer/6'
towncenter_url = r"https://maps.trpa.org/server/rest/services/Planning/FeatureServer/3"

#census_data = get_fs_data(census_url)
taz_data = get_fs_data_spatial(taz_url)
#census_geom_data = get_fs_data_spatial(census_geom_url)
towncenter_data = get_fs_data_spatial(towncenter_url)

local_path = pathlib.Path().absolute()
PROCESSED  = local_path.parents[0] / "data" / "processed_data"
TOTAL_JOBS = 270
OUTPUT_FILE_NAME = "taz_towncenter_jobs_alt_3.csv"

# ── 1. In-memory feature classes ─────────────────────────────────────────────
taz_mem = "memory/taz_tc"
tc_mem  = "memory/towncenter"

for fc in [taz_mem, tc_mem]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

taz_data.spatial.to_featureclass(location=taz_mem)
towncenter_data.spatial.to_featureclass(location=tc_mem)

print(f"TAZ features:        {arcpy.management.GetCount(taz_mem)[0]}")
print(f"Town-center features:{arcpy.management.GetCount(tc_mem)[0]}")

# ── 2. Spatial join – keep every TAZ that intersects a town center ────────────
sj_mem = "memory/taz_towncenter_join"
if arcpy.Exists(sj_mem):
    arcpy.management.Delete(sj_mem)

arcpy.analysis.SpatialJoin(
    target_features   = taz_mem,
    join_features     = tc_mem,
    out_feature_class = sj_mem,
    join_operation    = "JOIN_ONE_TO_ONE",
    join_type         = "KEEP_COMMON",
    match_option      = "INTERSECT",
)

tc_taz_set = {
    int(row[0])
    for row in arcpy.da.SearchCursor(sj_mem, ["TAZ"])
    if row[0] is not None
}
print(f"\nTAZs inside a town center: {len(tc_taz_set)}")
print(sorted(tc_taz_set))

# ── 3 & 4. Load 2050 SocioEcon, compare units, assign jobs ──────────────────
alt1 = pd.read_csv(PROCESSED / "Alternative_1_2050" / "SocioEcon_Summer.csv")[["taz", "total_residential_units"]]
alt2 = pd.read_csv(PROCESSED / "Alternative_2_reduced_2050" / "SocioEcon_Summer.csv")[["taz", "total_residential_units"]]

merged = alt1.merge(alt2, on="taz", suffixes=("_alt1", "_alt2"))

tc = merged[merged["taz"].isin(tc_taz_set)].copy()
tc["unit_diff"] = tc["total_residential_units_alt2"] - tc["total_residential_units_alt1"]
tc = tc[tc["unit_diff"] > 0].copy()

total_diff = tc["unit_diff"].sum()
if total_diff == 0:
    tc["additional_jobs"] = 0
else:
    exact = tc["unit_diff"] / total_diff * TOTAL_JOBS
    tc["additional_jobs"] = np.floor(exact).astype(int)

    remainder = TOTAL_JOBS - tc["additional_jobs"].sum()
    if remainder > 0:
        rng = np.random.default_rng(seed=42)
        chosen = rng.choice(tc.index, size=int(remainder), replace=False)
        tc.loc[chosen, "additional_jobs"] += 1

out = tc[["taz", "unit_diff", "additional_jobs"]].copy()

out_path = PROCESSED.parents[0] / "inputs" / OUTPUT_FILE_NAME
out_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(out_path, index=False)

print(f"\n→  {out_path.name}  ({len(out)} TAZs, total jobs assigned: {out['additional_jobs'].sum()})")
print(out.to_string(index=False))
print("\nDone.")
